# Laboratorio 02 — JOINs libres con PySpark

**Semana:** 02  
**Actividad de referencia:** Actividad 02  
**Estudiante:** Daniel Guzmán  
**Dataset:** Financial Transaction Dataset — Users + Cards  

## Parte 1 — Descripción del dataset

### Nombre y fuente

El dataset seleccionado hace parte del **Financial Transaction Dataset** usado durante la Semana 02 del bootcamp.

Para este laboratorio se usan dos archivos:

- `users_data.csv`
- `cards_data.csv`

Fuente: SharePoint del bootcamp  
Ruta: `inetum_data_engineer_bootcamp / semana_02 / financial_transaction_dataset`

### Dominio

El dominio del dataset es **financiero/bancario**.  
Describe clientes y tarjetas asociadas a esos clientes.

### ¿Por qué elegí este dataset?

Elegí estas tablas porque tienen una relación clara entre usuarios y tarjetas. Esto permite practicar JOINs y responder preguntas de negocio sobre distribución de tarjetas, límites de crédito y características de clientes.

### Tablas disponibles

| Tabla | Llave primaria | Llave foránea hacia |
|---|---|---|
| users_data | id | — |
| cards_data | id | client_id → users_data.id |

### Diagrama de relaciones

```text
users_data
   id
   │
   └── cards_data.client_id

In [0]:

## Celda 2 — Carga y perfil técnico




from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, sum as spark_sum

MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

df_users = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/users_data.csv")

df_cards = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/cards_data.csv")

for nombre, df in [("users", df_users), ("cards", df_cards)]:
    print(f"\n--- {nombre} ---")
    print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")
    df.printSchema()

In [0]:
print("=== Users ===")
df_users.describe().show(truncate=False)

print("=== Cards ===")
df_cards.describe().show(truncate=False)

In [0]:
def perfil_nulos(df, nombre):
    total = df.count()
    numeric_types = {"double", "float", "long", "integer", "short", "byte"}

    nulos = df.select([
        spark_sum(
            when(
                col(c).isNull() |
                (isnan(col(c)) if df.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
                (col(c).cast("string") == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    print(f"\n--- Nulos en {nombre} ({total:,} filas) ---")
    print(f"{'Columna':<35} {'Nulos':>8} {'%':>8}")
    print("-" * 54)

    for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
        if n > 0:
            print(f"{c:<35} {n:>8,} {n/total*100:>7.1f}%")

    if all(v == 0 for v in nulos.values()):
        print("Sin nulos detectados")

perfil_nulos(df_users, "users")
perfil_nulos(df_cards, "cards")

In [0]:
LLAVE_USERS = "id"
LLAVE_CARDS = "id"

dup_users = df_users.groupBy(LLAVE_USERS).count().filter(F.col("count") > 1).count()
dup_cards = df_cards.groupBy(LLAVE_CARDS).count().filter(F.col("count") > 1).count()

print(f"Duplicados en llave users.id: {dup_users:,}")
print(f"Duplicados en llave cards.id: {dup_cards:,}")

# Verificar si cards.client_id tiene usuarios asociados
cards_sin_usuario = df_cards.join(
    df_users.withColumnRenamed("id", "client_id"),
    on="client_id",
    how="left_anti"
)

print(f"Tarjetas sin usuario asociado: {cards_sin_usuario.count():,}")

## Observaciones del perfil técnico

Las tablas `users_data` y `cards_data` tienen una relación directa mediante `users_data.id = cards_data.client_id`.

No se encontraron duplicados en las llaves primarias:

- `users.id`: 0 duplicados.
- `cards.id`: 0 duplicados.

También se validó la integridad referencial entre tarjetas y usuarios. El resultado fue **0 tarjetas sin usuario asociado**, lo que indica que todos los registros de `cards_data.client_id` tienen correspondencia en `users_data.id`.

Esto es positivo para los JOINs, porque reduce el riesgo de pérdida de información o duplicación inesperada al unir las tablas.

In [0]:
# Parte 3 — Transformaciones previas al JOIN

# Preparar users
df_users_clean = (
    df_users
    .withColumnRenamed("id", "user_id")
    .withColumn("user_id", F.col("user_id").cast("int"))
    .withColumn("current_age", F.col("current_age").cast("int"))
    .withColumn("yearly_income_num", F.regexp_replace(F.col("yearly_income"), "[$,]", "").cast("double"))
    .withColumn("gender", F.trim(F.col("gender")))
)

# Preparar cards
df_cards_clean = (
    df_cards
    .withColumnRenamed("id", "card_id")
    .withColumnRenamed("client_id", "user_id")
    .withColumn("user_id", F.col("user_id").cast("int"))
    .withColumn("card_id", F.col("card_id").cast("int"))
    .withColumn("credit_limit_num", F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))
    .withColumn("card_type", F.trim(F.col("card_type")))
    .withColumn("card_brand", F.trim(F.col("card_brand")))
    .withColumn("has_chip", F.trim(F.col("has_chip")))
)

print("Users clean:")
df_users_clean.printSchema()

print("Cards clean:")
df_cards_clean.printSchema()

## Explicación de las transformaciones previas

Antes de unir las tablas se prepararon las llaves y columnas principales:

- En `users_data`, `id` se renombró a `user_id`.
- En `cards_data`, `client_id` se renombró a `user_id` y `id` a `card_id`.
- Las llaves `user_id` y `card_id` se castearón a entero para asegurar compatibilidad en los JOINs.
- `credit_limit` se limpió y convirtió a `credit_limit_num` para poder hacer análisis numéricos.
- `yearly_income` se limpió y convirtió a `yearly_income_num`.
- Se aplicó `trim()` a columnas categóricas para evitar diferencias por espacios.

Estas transformaciones ayudan a evitar JOINs incorrectos por diferencias de tipo o formato.

In [0]:
# JOIN 1 — INNER JOIN
# Une únicamente las tarjetas que tienen usuario asociado

users_originales = df_users_clean.count()
cards_originales = df_cards_clean.count()

df_inner = df_cards_clean.join(
    df_users_clean,
    on="user_id",
    how="inner"
)

print(f"Usuarios originales: {users_originales:,}")
print(f"Tarjetas originales: {cards_originales:,}")
print(f"Filas después del INNER JOIN: {df_inner.count():,}")
print(f"Tarjetas perdidas en INNER JOIN: {cards_originales - df_inner.count():,}")

display(df_inner.limit(5))

## Análisis JOIN 1 — INNER JOIN

El `INNER JOIN` conserva únicamente las tarjetas que tienen un usuario asociado en `users_data`.

Como previamente se validó que no había tarjetas sin usuario asociado, no se espera pérdida de registros. Este JOIN es útil cuando se necesita trabajar solo con registros que tienen correspondencia completa entre ambas tablas.

In [0]:
# JOIN 2 — LEFT JOIN
# Conserva todas las tarjetas, aunque alguna no tuviera usuario asociado

df_left = df_cards_clean.join(
    df_users_clean,
    on="user_id",
    how="left"
)

sin_usuario = df_left.filter(F.col("gender").isNull()).count()

print(f"Tarjetas originales: {cards_originales:,}")
print(f"Filas después del LEFT JOIN: {df_left.count():,}")
print(f"Tarjetas sin datos de usuario después del LEFT JOIN: {sin_usuario:,}")

display(df_left.limit(5))

In [0]:
# JOIN 2 — LEFT JOIN
# Conserva todas las tarjetas, aunque alguna no tuviera usuario asociado

df_left = df_cards_clean.join(
    df_users_clean,
    on="user_id",
    how="left"
)

sin_usuario = df_left.filter(F.col("gender").isNull()).count()

print(f"Tarjetas originales: {cards_originales:,}")
print(f"Filas después del LEFT JOIN: {df_left.count():,}")
print(f"Tarjetas sin datos de usuario después del LEFT JOIN: {sin_usuario:,}")

display(df_left.limit(5))

## Análisis JOIN 2 — LEFT JOIN

El `LEFT JOIN` conserva todas las tarjetas de `cards_data`, incluso si alguna no tuviera usuario asociado.

En este dataset no se encontraron tarjetas sin datos de usuario, lo que confirma una buena integridad referencial entre `cards_data.client_id` y `users_data.id`.

En un caso real, este JOIN sería útil para detectar registros huérfanos sin perder la tabla principal de tarjetas.

In [0]:
# JOIN 3 — Anti JOIN
# Busca tarjetas cuyo user_id no exista en users

df_cards_sin_usuario = df_cards_clean.join(
    df_users_clean,
    on="user_id",
    how="left_anti"
)

print(f"Tarjetas sin usuario asociado: {df_cards_sin_usuario.count():,}")

display(df_cards_sin_usuario.limit(10))

In [0]:
# JOIN 3 — Anti JOIN
# Busca tarjetas cuyo user_id no exista en users

df_cards_sin_usuario = df_cards_clean.join(
    df_users_clean,
    on="user_id",
    how="left_anti"
)

print(f"Tarjetas sin usuario asociado: {df_cards_sin_usuario.count():,}")

display(df_cards_sin_usuario.limit(10))

## Análisis JOIN 3 — Anti JOIN

El `left_anti` permite encontrar registros de la tabla izquierda que no tienen pareja en la tabla derecha.

En este caso se usó para identificar tarjetas sin usuario asociado. El resultado fue 0, lo que indica que no hay problemas de integridad referencial entre las dos tablas usadas.

Este tipo de JOIN es muy útil como prueba de calidad de datos antes de construir una tabla final enriquecida.

In [0]:
# Pregunta 1: ¿Qué tipo de tarjeta es más frecuente por género del cliente?

df_tipo_por_genero = (
    df_left
    .groupBy("gender", "card_type")
    .agg(
        F.count("card_id").alias("total_tarjetas"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio")
    )
    .orderBy("gender", F.col("total_tarjetas").desc())
)

df_tipo_por_genero.show(truncate=False)

## Conclusión pregunta 1

El tipo de tarjeta más frecuente tanto en mujeres como en hombres es **Debit**.

En clientes **Female**, hay **1,814 tarjetas Debit**, seguidas por **1,043 Credit** y **282 Debit (Prepaid)**.  
En clientes **Male**, hay **1,697 tarjetas Debit**, seguidas por **1,014 Credit** y **296 Debit (Prepaid)**.

Esto muestra que la tarjeta débito domina en ambos grupos, sin una diferencia fuerte en la distribución por género. También se observa que las tarjetas Debit tienen límites promedio más altos que las Credit dentro de ambos géneros.

In [0]:
# Pregunta 2: ¿Qué grupo de clientes tiene mayor límite de crédito promedio?

df_grupo_edad = (
    df_left
    .withColumn(
        "grupo_edad",
        F.when(F.col("current_age") < 30, "menor_30")
         .when(F.col("current_age") < 45, "30_44")
         .when(F.col("current_age") < 60, "45_59")
         .otherwise("60_mas")
    )
    .groupBy("grupo_edad")
    .agg(
        F.count("card_id").alias("total_tarjetas"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio"),
        F.max("credit_limit_num").alias("limite_maximo"),
        F.round(F.avg("yearly_income_num"), 2).alias("ingreso_promedio")
    )
    .orderBy(F.col("limite_promedio").desc())
)

df_grupo_edad.show(truncate=False)

## Conclusión pregunta 2

El grupo de edad con mayor límite de crédito promedio es **45_59**, con un límite promedio de **14,998.05**.

Le siguen los grupos **30_44** con **14,114.81**, **menor_30** con **14,060.36** y **60_mas** con **14,057.56**.

Aunque las diferencias no son extremadamente grandes, el grupo de 45 a 59 años parece tener el perfil crediticio más alto dentro del dataset. También es el grupo con mayor ingreso promedio, con **47,792.83**, lo que puede explicar parte de ese mayor límite.

In [0]:
# Pregunta 3: ¿Qué usuarios tienen más tarjetas asociadas?

df_usuarios_mas_tarjetas = (
    df_left
    .groupBy("user_id", "gender", "current_age", "yearly_income_num")
    .agg(
        F.count("card_id").alias("total_tarjetas"),
        F.round(F.sum("credit_limit_num"), 2).alias("limite_total"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio")
    )
    .orderBy(F.col("total_tarjetas").desc(), F.col("limite_total").desc())
)

df_usuarios_mas_tarjetas.show(10, truncate=False)

## Conclusión pregunta 3

Los usuarios con más tarjetas asociadas tienen entre **8 y 9 tarjetas**.

El usuario **797**, mujer de **61 años**, aparece con **9 tarjetas** y un límite total de **253,066.0**. También el usuario **1301**, mujer de **26 años**, tiene **9 tarjetas** y un límite total de **246,947.0**.

Este análisis permite identificar clientes con alta exposición crediticia. En un entorno financiero, estos usuarios podrían ser relevantes para análisis de riesgo, segmentación comercial o monitoreo de uso de productos.

## Parte 6 — Reflexión final

### ¿Qué tipo de JOIN fue el más revelador para entender tu dataset?

El `left_anti` fue el más revelador para validar la calidad del modelo, porque permitió confirmar si existían tarjetas sin usuario asociado. El resultado fue 0, lo que indica que no se encontraron problemas de integridad referencial entre `cards_data.client_id` y `users_data.id`.

### ¿Encontraste algún problema de integridad referencial? ¿Qué implica?

No se encontraron problemas de integridad referencial entre usuarios y tarjetas. Todas las tarjetas tenían un usuario válido asociado.

Esto implica que los JOINs entre ambas tablas se pueden hacer con confianza, sin pérdida inesperada de registros por llaves inexistentes.

### ¿Qué columna del DataFrame unido resultó ser la más útil para el análisis?

La columna `credit_limit_num` fue una de las más útiles, porque permitió calcular límites promedio, límites totales y comparar grupos de clientes por género, edad y número de tarjetas.

También fueron útiles `card_type`, `gender` y `current_age`, porque permitieron segmentar el análisis.

### Si fueras a construir un pipeline de producción con este dataset, ¿qué validaciones harías antes de los JOINs?

Antes de los JOINs validaría:

- Que las llaves `users.id` y `cards.client_id` tengan el mismo tipo de dato.
- Que no existan duplicados en las llaves primarias.
- Que no existan nulos en columnas clave.
- Que todas las tarjetas tengan un usuario asociado.
- Que columnas financieras como `credit_limit` puedan convertirse correctamente a tipo numérico.
- Que las columnas categóricas no tengan espacios extra o valores inconsistentes.